# V12 Leave one platform out

Setiap platform menjadi test pada eksperimen terpisah. Training hanya menggunakan platform lain. Output otomatis masuk folder run dengan akhiran `_lopo`.

**Sebelum mulai:** jalankan `00_persiapan_runtime.ipynb`, lalu pilih **Runtime → Restart session** sekali. Notebook ini tidak memasang ulang paket.

Ekstrak seluruh folder `reviewer_v12` ke Drive. Notebook dibaca dari atas ke bawah; implementasi lengkap berada di file `.py` yang menyertainya.

## 1. Hubungkan Google Drive

Output yang diharapkan: Drive terpasang pada `/content/drive`.

In [1]:
try:
    from google.colab import drive
except ImportError:
    print("Runtime lokal: gunakan path lokal di konfigurasi berikut.")
else:
    drive.mount("/content/drive")

Mounted at /content/drive


## 2. Tentukan folder dan run

DATA_DIR boleh di v3, sedangkan VIDEO_DIR tetap di v2. Gunakan RUN_NAME yang sama antar notebook utama dan baseline. Run baru ini terpisah dari hasil paket lama.

In [2]:
# 1. Folder reviewer_v12 yang berisi file-file .py
PACKAGE_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/Collab/reviewer_v12"

# 2. Folder data yang berisi final_labeled_dataset_v5_clean.csv
DATA_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/data"

# 3. Folder yang berisi 579 file video mp4/avi
VIDEO_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber Anak v2/scraped_dataset/videos"

# 4. Biarkan apa adanya (jangan diubah)
RUN_NAME = "reviewer_v12_20265495_fix1"

# 5. None = menjalankan semua platform LOPO
ONLY_EXPERIMENTS = None


## 3. Temukan file kode pendukung

Tidak ada kode model yang ditumpuk di sel ini. Bila lokasi tidak tunggal, isi PACKAGE_DIR pada tahap 2.

In [3]:
import os
from pathlib import Path
import sys

# Hilangkan spasi/kutip dan JANGAN gunakan .resolve() yang merusak shortcut Drive
pkg_str = PACKAGE_DIR.strip().strip("'\"") if PACKAGE_DIR else ""
package_path = Path(pkg_str) if pkg_str else None

if package_path and (package_path / "runtime_setup.py").is_file():
  PACKAGE = package_path
elif (
    package_path and (package_path / "reviewer_v12" / "runtime_setup.py").is_file()
):
  PACKAGE = package_path / "reviewer_v12"
else:
  # Cari otomatis di Drive jika path shortcut belum tepat
  roots = [
      Path("/content/drive/MyDrive"),
      Path("/content/drive/.shortcut-targets-by-id"),
      Path.cwd(),
  ]
  found = []
  for base in roots:
    if not base.exists():
      continue
    for folder, dirs, files in os.walk(base, followlinks=False):
      if "runtime_setup.py" in files and "revision_session.py" in files:
        found.append(Path(folder))
        dirs[:] = []
      else:
        depth = len(Path(folder).relative_to(base).parts)
        dirs[:] = [
            d
            for d in dirs
            if depth < 8
            and d
            not in {
                "scraped_dataset",
                "revision_runs",
                "output",
                ".git",
                ".ipynb_checkpoints",
            }
        ]

  found = list(dict.fromkeys(found))
  if len(found) >= 1:
    v12 = [p for p in found if p.name == "reviewer_v12"]
    PACKAGE = v12[0] if v12 else found[0]
  else:
    raise FileNotFoundError(
        "File 'runtime_setup.py' belum ditemukan di Google Drive akun ini!"
    )

PACKAGE_DIR = str(PACKAGE)
if str(PACKAGE) not in sys.path:
  sys.path.insert(0, str(PACKAGE))

print(" Paket berhasil ditemukan dan siap:", PACKAGE)


 Paket berhasil ditemukan dan siap: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/Collab/reviewer_v12


## 4. Periksa runtime aktif

Pemeriksaan menguji NumPy strings, SciPy sparse, dan scikit-learn sebelum impor pipeline. Jika diminta restart, lakukan restart dan jalankan dari tahap 1.

In [4]:
from runtime_setup import verify_current_process

verify_current_process(require_training=True)

{"python": "3.13.15", "numpy": "2.1.3", "scipy": "1.16.3", "pandas": "2.2.3", "sklearn": "1.6.1"}
GPU siap: Tesla T4
Pemeriksaan runtime aktif LULUS.


True

## 5. Baca data dan tampilkan hitungan

Output: jumlah raw, clean, transkrip yang cocok, text-only, dan kelompok pembagian data. File input tidak ditimpa.

In [5]:
from revision_session import ExperimentSession

session = ExperimentSession(
    data_dir=DATA_DIR,
    video_dir=VIDEO_DIR,
    run_name=RUN_NAME,
    only=ONLY_EXPERIMENTS,
    lopo=True,
)
display(session.data_summary())

Data: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/data
Output: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1_lopo


,Pemeriksaan,Jumlah
0,Baris raw,15046
1,Baris clean,15044
2,Transkrip nonkosong yang cocok,450
3,Text-only nominal pada raw,13962
4,Kelompok untuk split,3585


## 6. Bekukan konfigurasi dan split

Output: tabel jumlah sampel setiap role. Threshold dan epoch dipilih hanya dari partisi internal; test tidak digunakan untuk memilihnya.

In [6]:
display(session.prepare_splits())

role,calibration,early_stop,fit,meta_fit,test
experiment,,,,,
lopo_facebook,2117,1091,9012,2120,704
lopo_instagram,2159,968,8870,2170,877
lopo_twitter,2078,1001,8787,2097,1081
lopo_youtube,401,185,1675,401,12382


## 7. Periksa media yang benar-benar terbaca

Output: hitungan coverage aktual. Nama video dicocokkan persis dengan ID; tidak ada pemaksaan jumlah visual menjadi 566.

In [7]:
import json
from pathlib import Path
import shutil
import pandas as pd

# Ambil hasil pengecekan media yang sudah selesai di run utama (agar tidak scan ulang 9GB video)
src = session.out.parent / session.out.name.replace("_lopo", "")
dest = session.out
dest.mkdir(parents=True, exist_ok=True)

for fname in ["modality_masks.csv", "coverage.json", "media_signature.json"]:
  if (src / fname).exists():
    shutil.copy2(src / fname, dest / fname)

masks = pd.read_csv(dest / "modality_masks.csv")
session.visual_mask = session.core.bool_column(masks.visual_effective)
coverage = json.loads((dest / "coverage.json").read_text())

print(" Berhasil mengambil cache media tanpa membebani jaringan Drive!")
display(coverage)


 Berhasil mengambil cache media tanpa membebani jaringan Drive!


{'n': 15044,
 'transcript_real': 450,
 'visual_real': 565,
 'both_real': 0,
 'neither_auxiliary_real': 14029,
 'video_files_matched': 579,
 'video_directory_files': 579,
 'audio_status': 'existing transcript CSV; ASR provenance/quality not independently verified',
 'visual_status': 'actual decoding in this run, not copied from paper'}

## 8. Lihat status sebelum training

“Selesai” berarti probabilitas cabang sudah tersimpan. Menjalankan sel yang sama lagi akan memakai hasil yang sesuai run.

In [8]:
display(session.progress())

,experiment,indobert,indobertweet,mbert,transcript,visual
0,lopo_facebook,selesai,selesai,selesai,selesai,belum
1,lopo_instagram,selesai,selesai,selesai,selesai,belum
2,lopo_twitter,selesai,selesai,selesai,selesai,belum
3,lopo_youtube,selesai,selesai,selesai,selesai,belum


## 9. Latih IndoBERT

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [9]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "indobert" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("indobert*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — indobert")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/indobert E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



lopo_facebook — indobert
lopo_facebook/indobert E1: loss=0.05559 inner_F1=0.85373
lopo_facebook/indobert E2: loss=0.03467 inner_F1=0.88709
lopo_facebook/indobert E3: loss=0.02203 inner_F1=0.88828
lopo_facebook/indobert E4: loss=0.01375 inner_F1=0.88764
lopo_facebook/indobert E5: loss=0.00924 inner_F1=0.90038
lopo_facebook/indobert E6: loss=0.00631 inner_F1=0.90133
lopo_facebook/indobert E7: loss=0.00444 inner_F1=0.90215
lopo_facebook/indobert E8: loss=0.00394 inner_F1=0.89942
lopo_facebook/indobert E9: loss=0.00393 inner_F1=0.90028
lopo_facebook/indobert E10: loss=0.00465 inner_F1=0.89373

lopo_instagram — indobert
lopo_instagram/indobert E1: loss=0.06023 inner_F1=0.85831

lopo_twitter — indobert
lopo_twitter/indobert E1: loss=0.05856 inner_F1=0.88532
lopo_twitter/indobert E2: loss=0.03542 inner_F1=0.91350
lopo_twitter/indobert E3: loss=0.02318 inner_F1=0.91075
lopo_twitter/indobert E4: loss=0.01529 inner_F1=0.90533
lopo_twitter/indobert E5: loss=0.01092 inner_F1=0.90947

lopo_youtube

,experiment,indobert,indobertweet,mbert,transcript,visual
0,lopo_facebook,selesai,selesai,selesai,selesai,belum
1,lopo_instagram,selesai,selesai,selesai,selesai,belum
2,lopo_twitter,selesai,selesai,selesai,selesai,belum
3,lopo_youtube,selesai,selesai,selesai,selesai,belum


In [10]:
import importlib, shutil, subprocess, torch
from pathlib import Path

# ─── Reload modul dari Drive (ambil versi terbaru dengan tqdm) ───
import revision_train as _rt
importlib.reload(_rt)

DRIVE_PFX = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"
MIRROR = Path("/tmp/lopo_mirror")

# ─── Restore torch.load ke asli ───
import torch.serialization as _ts
_REAL_LOAD = _ts.load
torch.load = _REAL_LOAD
_rt.torch.load = _REAL_LOAD

# ─── Mirror helper ───
def _to_mirror(p):
    s = str(p)
    return MIRROR / s[len(DRIVE_PFX):].lstrip("/") if s.startswith(DRIVE_PFX) else None

# ─── Patch atomic_torch ───
def safe_atomic_torch(path, obj):
    dest = _to_mirror(path) or Path(path)
    dest.parent.mkdir(parents=True, exist_ok=True)
    torch.save(obj, str(dest))

def safe_torch_load(f, *args, **kwargs):
    try:
        m = _to_mirror(Path(str(f)))
        if m and m.exists():
            return _REAL_LOAD(str(m), *args, **kwargs)
    except Exception:
        pass
    return _REAL_LOAD(f, *args, **kwargs)

_rt.atomic_torch = safe_atomic_torch
_rt.torch.load = safe_torch_load
torch.load = safe_torch_load

# ─── Patch run_branch: training → /tmp, predictions.npz → copy ke Drive ───
_orig_run_branch = _rt.run_branch

def fully_mirrored_run_branch(df, roles, branch, config, folder, cache_dir, visual_mask, seed, device):
    folder = Path(str(folder))
    if (folder / "predictions.npz").exists():
        return _orig_run_branch(df, roles, branch, config, str(folder), cache_dir, visual_mask, seed, device)
    mirror = _to_mirror(folder)
    if mirror is None:
        return _orig_run_branch(df, roles, branch, config, str(folder), cache_dir, visual_mask, seed, device)
    mirror.mkdir(parents=True, exist_ok=True)
    result = _orig_run_branch(df, roles, branch, config, str(mirror), cache_dir, visual_mask, seed, device)
    subprocess.run(["mkdir", "-p", str(folder)], check=False, capture_output=True)
    for src in mirror.rglob("*"):
        if src.is_file():
            rel = src.relative_to(mirror)
            dst = folder / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            try:
                shutil.copy(str(src), str(dst))
            except Exception as e:
                print("GAGAL salin:", rel, e)
    return result

_rt.run_branch = fully_mirrored_run_branch
print("Reload + Patch aktif. Progress bar per-batch sekarang aktif!")


Reload + Patch aktif. Progress bar per-batch sekarang aktif!


## 10. Latih IndoBERTweet

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [11]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "indobertweet" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("indobertweet*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — indobertweet")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/indobertweet E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



lopo_facebook — indobertweet

lopo_instagram — indobertweet
lopo_instagram/indobertweet E1: loss=0.06987 inner_F1=0.79815
lopo_instagram/indobertweet E2: loss=0.04770 inner_F1=0.83968
lopo_instagram/indobertweet E3: loss=0.03754 inner_F1=0.84895
lopo_instagram/indobertweet E4: loss=0.03057 inner_F1=0.85070
lopo_instagram/indobertweet E5: loss=0.02483 inner_F1=0.85704
lopo_instagram/indobertweet E6: loss=0.02000 inner_F1=0.85946
lopo_instagram/indobertweet E7: loss=0.01741 inner_F1=0.85604
lopo_instagram/indobertweet E8: loss=0.01568 inner_F1=0.85594
lopo_instagram/indobertweet E9: loss=0.01444 inner_F1=0.85875

lopo_twitter — indobertweet
lopo_twitter/indobertweet E1: loss=0.07396 inner_F1=0.82803
lopo_twitter/indobertweet E2: loss=0.04654 inner_F1=0.87786
lopo_twitter/indobertweet E3: loss=0.03643 inner_F1=0.89879
lopo_twitter/indobertweet E4: loss=0.02911 inner_F1=0.88195
lopo_twitter/indobertweet E5: loss=0.02376 inner_F1=0.89144
lopo_twitter/indobertweet E6: loss=0.02000 inner_F1=

,experiment,indobert,indobertweet,mbert,transcript,visual
0,lopo_facebook,selesai,selesai,selesai,selesai,belum
1,lopo_instagram,selesai,selesai,selesai,selesai,belum
2,lopo_twitter,selesai,selesai,selesai,selesai,belum
3,lopo_youtube,selesai,selesai,selesai,selesai,belum


## 11. Latih multilingual BERT

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [12]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "mbert" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("mbert*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — mbert")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/mbert E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



lopo_facebook — mbert
lopo_facebook/mbert E1: loss=0.07712 inner_F1=0.73965

lopo_instagram — mbert

lopo_twitter — mbert

lopo_youtube — mbert


,experiment,indobert,indobertweet,mbert,transcript,visual
0,lopo_facebook,selesai,selesai,selesai,selesai,belum
1,lopo_instagram,selesai,selesai,selesai,selesai,belum
2,lopo_twitter,selesai,selesai,selesai,selesai,belum
3,lopo_youtube,selesai,selesai,selesai,selesai,belum


## 12. Latih cabang transkrip

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [13]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "transcript" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("transcript*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — transcript")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/transcript E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



lopo_facebook — transcript
lopo_facebook/transcript E1: loss=0.08092 inner_F1=0.80458
lopo_facebook/transcript E2: loss=0.04977 inner_F1=0.84777
lopo_facebook/transcript E3: loss=0.03888 inner_F1=0.86050
lopo_facebook/transcript E4: loss=0.03113 inner_F1=0.85831
lopo_facebook/transcript E5: loss=0.02568 inner_F1=0.86305
lopo_facebook/transcript E6: loss=0.02155 inner_F1=0.86709
lopo_facebook/transcript E7: loss=0.01785 inner_F1=0.86750
lopo_facebook/transcript E8: loss=0.01648 inner_F1=0.87671
lopo_facebook/transcript E9: loss=0.01518 inner_F1=0.87368
lopo_facebook/transcript E10: loss=0.01509 inner_F1=0.86979

lopo_instagram — transcript
lopo_instagram/transcript E1: loss=0.07752 inner_F1=0.80759
lopo_instagram/transcript E2: loss=0.05205 inner_F1=0.83984
lopo_instagram/transcript E3: loss=0.04095 inner_F1=0.85490
lopo_instagram/transcript E4: loss=0.03373 inner_F1=0.84983
lopo_instagram/transcript E5: loss=0.02767 inner_F1=0.87061
lopo_instagram/transcript E6: loss=0.02323 inner_F1=

,experiment,indobert,indobertweet,mbert,transcript,visual
0,lopo_facebook,selesai,selesai,selesai,selesai,belum
1,lopo_instagram,selesai,selesai,selesai,selesai,belum
2,lopo_twitter,selesai,selesai,selesai,selesai,belum
3,lopo_youtube,selesai,selesai,selesai,selesai,belum


## 13. Latih EfficientNet-B4

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [22]:
import json
from pathlib import Path

MIRROR = Path("/tmp/lopo_mirror")
DRIVE_PFX = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"

for exp in session._selected_experiments():
    print(f"\n{exp} — visual")
    drive_h = session.out / exp / "visual" / "history.json"
    rel = str(drive_h).replace(DRIVE_PFX, "").lstrip("/")
    mirror_h = MIRROR / rel
    h_file = drive_h if drive_h.exists() else (mirror_h if mirror_h.exists() else None)
    if h_file:
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/visual E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")
    else:
        roles = session.splits[exp]
        if len([i for i in roles["fit"] if session.visual_mask[i]]) == 0:
            print(f"{exp}/visual: tidak ada video di fit split (platform test-only) — tidak dilatih")
        else:
            print(f"{exp}/visual: history.json tidak tersedia")



lopo_facebook — visual
lopo_facebook/visual E1: loss=0.08478 inner_F1=0.63135
lopo_facebook/visual E2: loss=0.08047 inner_F1=0.60256
lopo_facebook/visual E3: loss=0.07437 inner_F1=0.60256
lopo_facebook/visual E4: loss=0.06723 inner_F1=0.61250

lopo_instagram — visual
lopo_instagram/visual E1: loss=0.08580 inner_F1=0.58365
lopo_instagram/visual E2: loss=0.08200 inner_F1=0.60758
lopo_instagram/visual E3: loss=0.07658 inner_F1=0.63142
lopo_instagram/visual E4: loss=0.07093 inner_F1=0.69467
lopo_instagram/visual E5: loss=0.06565 inner_F1=0.72953
lopo_instagram/visual E6: loss=0.06266 inner_F1=0.75388
lopo_instagram/visual E7: loss=0.06047 inner_F1=0.78363
lopo_instagram/visual E8: loss=0.05903 inner_F1=0.75019

lopo_twitter — visual
lopo_twitter/visual E1: loss=0.08531 inner_F1=0.36979
lopo_twitter/visual E2: loss=0.08247 inner_F1=0.56796
lopo_twitter/visual E3: loss=0.07664 inner_F1=0.56617
lopo_twitter/visual E4: loss=0.07265 inner_F1=0.54451
lopo_twitter/visual E5: loss=0.06958 inner_F

## 14. Jalankan fusion dan evaluasi

Jalankan setelah lima cabang selesai. LR/MLP, threshold, statistik berpasangan, dan tabel dihitung dari probabilitas tersimpan. Bila masih ada cabang belum selesai, namanya ditampilkan.

In [30]:
import importlib, shutil, subprocess, numpy as np
from pathlib import Path
import revision_train as _rt
importlib.reload(_rt)

MIRROR     = Path("/tmp/lopo_mirror")
FUSION_RUN = Path("/tmp/fusion_run_lopo")
DRIVE_PFX  = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"

for exp in session._selected_experiments():
    for b in session.core.BRANCHES:
        dst = FUSION_RUN / exp / b / "predictions.npz"
        if dst.exists() and dst.stat().st_size > 10000:
            continue
        rel = str(session.out / exp / b).replace(DRIVE_PFX, "").lstrip("/")
        found = False
        for src in [MIRROR / rel / "predictions.npz", session.out / exp / b / "predictions.npz"]:
            if src.exists() and src.stat().st_size > 10000:
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(str(src), str(dst))
                print(f"Ready: {exp}/{b}")
                found = True
                break
        if not found and b == "visual":
            roles = session.splits[exp]
            if len([i for i in roles["fit"] if session.visual_mask[i]]) == 0:
                ids = np.concatenate([roles[k] for k in ["meta_fit", "calibration", "test"]])
                bounds = np.cumsum([0] + [len(roles[k]) for k in ["meta_fit", "calibration", "test"]])
                dst.parent.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(
                    str(dst),
                    ids=ids,
                    prob=np.full(len(ids), 0.5, dtype=np.float32),
                    features=np.zeros((len(ids), 1792), dtype=np.float32),
                    bounds=bounds,
                )
                print(f"Dummy: {exp}/{b}")

_rt.run_experiments(
    session.df, session.splits, FUSION_RUN,
    session.cache_dir, session.visual_mask, session.config,
    only=session.only,
)

for f in FUSION_RUN.rglob("*"):
    if f.is_file() and "predictions.npz" not in f.name:
        rel = f.relative_to(FUSION_RUN)
        dst = session.out / rel
        subprocess.run(["mkdir", "-p", str(dst.parent)], check=False)
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy(str(f), str(dst))
        except Exception as e:
            print(f"Gagal: {rel}: {e}")

display(session.progress())


Ready: lopo_facebook/indobert
Ready: lopo_facebook/indobertweet
Ready: lopo_facebook/mbert
Ready: lopo_facebook/transcript
Ready: lopo_facebook/visual
Ready: lopo_instagram/indobert
Ready: lopo_instagram/indobertweet
Ready: lopo_instagram/mbert
Ready: lopo_instagram/transcript
Ready: lopo_instagram/visual
Ready: lopo_twitter/indobert
Ready: lopo_twitter/indobertweet
Ready: lopo_twitter/mbert
Ready: lopo_twitter/transcript
Ready: lopo_twitter/visual
Ready: lopo_youtube/indobert
Ready: lopo_youtube/indobertweet
Ready: lopo_youtube/mbert
Ready: lopo_youtube/transcript
Dummy: lopo_youtube/visual


,experiment,indobert,indobertweet,mbert,transcript,visual
0,lopo_facebook,selesai,selesai,selesai,selesai,belum
1,lopo_instagram,selesai,selesai,selesai,selesai,belum
2,lopo_twitter,selesai,selesai,selesai,selesai,belum
3,lopo_youtube,selesai,selesai,selesai,selesai,belum


## 15. Tampilkan tabel hasil

Skala metrik adalah 0–1, termasuk average precision. Laporan final dibuat setelah seluruh partisi selesai.

In [31]:
display(session.results())

,model,n,accuracy,macro_f1,roc_auc,average_precision
0,indobert,13307,0.774480,0.765225,0.864692,0.755744
1,indobertweet,13307,0.766664,0.752266,0.830690,0.704959
2,mbert,13307,0.686180,0.656176,0.716876,0.555048
3,transcript,13307,0.750282,0.729309,0.798226,0.634629
4,visual,13307,0.653265,0.395136,0.500000,0.346735
5,text_mean,13307,0.796197,0.758678,0.857387,0.727371
6,text_lr,13307,0.797700,0.759576,0.860951,0.749497
7,text_speech_lr,13307,0.805967,0.775660,0.860434,0.748143
8,text_visual_lr,13307,0.797851,0.759789,0.860968,0.749596
9,speech_visual_lr,13307,0.752611,0.730558,0.792178,0.641235


## 16. Ekspor bahan revisi

ZIP berisi prediksi, pembagian sampel, masks, threshold, log, tabel, dan grafik. Bobot besar serta media mentah tidak dimasukkan.

In [32]:
zip_hasil = session.export()
print("Kirim ZIP hasil ini beserta notebook yang sudah dijalankan.")

ZIP hasil: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1_lopo_RESULTS_FOR_REVIEW.zip
Kirim ZIP hasil ini beserta notebook yang sudah dijalankan.


**Berikutnya:** kirim ZIP LOPO bersama ZIP utama setelah baseline.